In [1]:
# !/usr/bin/env python3
import os
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    spherical_grid, spherical_radial_sampling
import equiv_dens.utils.base as utils
from equiv_dens.training.model_loader import load_model

import numpy as np
from functools import partial
import argparse
%load_ext autoreload
%autoreload 2

Use "numpy" for Fourier Transform


/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


In [12]:
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_001_mae_test.txt')

restart=None
print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified
directory = args.restart  # load directory name
# load latest checkpoint
checkpoint_path = os.path.join(directory, 'checkpoints')  # checkpoint directory
checkpoint = torch.load(os.path.join(
    checkpoint_path, 'latest_checkpoint.pth'), map_location='cpu')
latest_checkpoint = checkpoint['step']
model_code = checkpoint['ID']  # load ID
step = checkpoint['step']
for arg in vars(checkpoint['args']):
    if args.fix_arguments:
        if arg in hyperparam_args:
            print('loading hyperparam arg', arg)
            setattr(args, arg, getattr(checkpoint['args'], arg))
    else:
        print('loading all arg', arg)
        setattr(args, arg, getattr(checkpoint['args'], arg))
restore = True

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = False
# load dataset(s)
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")

args.verbose = 0
args.use_gpu = False
args.radii_adjust = True 
if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None
    
dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose,
                           radii_adjust=args.radii_adjust)


type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
loading hyperparam arg activation
loading hyperparam arg order
loading hyperparam arg mixing_order
loading hyperparam arg order_en
loading hyperparam arg mixing_order_en
loading hyperparam arg num_features
loading hyperparam arg num_basis_functions
loading hyperparam arg num_radial_components
loading hyperparam arg num_energy_features
loading hyperparam arg num_modules
loading hyperparam arg num_residual_pre_x
loading hyperparam arg num_residual_post_x
loading hyperparam arg num_residual_pre_vi
loading hyperparam arg num_residual_pre_vj
loading hyperparam arg num_residual_post_v
loading hyperparam arg num_residual_output
loading hyperparam arg num_energy_output
loading hyperparam arg basis_functions
loading hyperparam arg cutoff
loading hyperparam arg orthonormal_basis
loading hyperparam arg expansion_constraint
loading hyperparam arg integral_constraint
loading hyperparam arg integral_scale
loading hyperparam 

In [14]:
args.restart = None
args.density_integral = False
model = load_model(args, dataset)

sample = dataset.get_properties([5])
res = model(sample)

print('density integral', torch.sum(sample['density'] * sample['coord_weights']))
print('pred density integral', torch.sum(res['density'] * sample['coord_weights']))
print('density loss', torch.sum(torch.abs(sample['density'] - res['density']) * sample['coord_weights'])/torch.sum(sample['atom_numbers']))

torch.float32
cg_matrix shape torch.Size([121, 121, 121])
args energy_unit_in kcal
args energy_unit_out kcal
conversions in <function kcal_to_kcal at 0x7f19934c98c8>
conversions out <function kcal_to_kcal at 0x7f19934c98c8>
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
creating embedding
init_coeffs None
orbital basis {6: [(6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 4), (6, 1, 4), (6, 1, 4), (6, 1, 5), (6, 1, 5)], 8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8

In [39]:
args.restart = None
args.integral_constraint = None 
args.expansion_constraint = None
args.verbose = 0
model = load_model(args, dataset)

sample = dataset.get_properties([5])
res = model(sample)

print('density integral', torch.sum(sample['density'] * sample['coord_weights']))
print('pred density integral', torch.sum(res['density'] * sample['coord_weights']))
print('density loss', torch.sum(torch.abs(sample['density'] - res['density']) * sample['coord_weights'])/torch.sum(sample['atom_numbers']))

cg_matrix shape torch.Size([121, 121, 121])
args energy_unit_in kcal
args energy_unit_out kcal
conversions in <function kcal_to_kcal at 0x7f19934c98c8>
conversions out <function kcal_to_kcal at 0x7f19934c98c8>
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
creating embedding
init_coeffs None
orbital basis {6: [(6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 4), (6, 1, 4), (6, 1, 4), (6, 1, 5), (6, 1, 5)], 8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1

In [66]:
args.restart = None
args.integral_constraint = None 
args.scale_sph_degrees = True
args.expansion_constraint = None 
args.verbose = 2
args.L0_coeffs_file = 'datasets/augccpvqzjkfit_init_L0.npy'
model = load_model(args, dataset)

sample = dataset.get_properties([5])
res = model(sample)

print('density integral', torch.sum(sample['density'] * sample['coord_weights']))
print('pred density integral', torch.sum(res['density'] * sample['coord_weights']))
print('density loss', torch.sum(torch.abs(sample['density'] - res['density']) * sample['coord_weights'])/torch.sum(sample['atom_numbers']))

cg_matrix shape torch.Size([121, 121, 121])
args energy_unit_in kcal
args energy_unit_out kcal
conversions in <function kcal_to_kcal at 0x7f19934c98c8>
conversions out <function kcal_to_kcal at 0x7f19934c98c8>
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
creating embedding
init_coeffs None
orbital basis {6: [(6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 4), (6, 1, 4), (6, 1, 4), (6, 1, 5), (6, 1, 5)], 8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1

In [68]:
args.restart = None
args.integral_constraint = None 
args.scale_sph_degrees = True
args.expansion_constraint = None 
args.verbose = 0
args.L0_coeffs_file = 'datasets/augccpvqzjkfit_init_L0.npy'
L0_coeffs = np.load(args.L0_coeffs_file, allow_pickle=True).item()
model = load_model(args, dataset)
ml_integral = []
ml_loss = []

for i in range(10):
    sample = dataset.get_properties([i])
    res = model(sample)

    for i in range(len(res['spherical_coeffs'])):
        for key in res['spherical_coeffs'][i].keys():
            z = key[0]
            z_s = utils.numbers_to_symbols([z])[0]
            L = key[1]
            if L == 0:
                res['spherical_coeffs'][i][key] = L0_coeffs['spherical_coeffs'][z_s].unsqueeze(0).unsqueeze(0)
                res['radial_width'][i][key] = L0_coeffs['radial_width'][z_s].unsqueeze(0).unsqueeze(0)
                res['radial_scale'][i][key] = L0_coeffs['radial_scale'][z_s].unsqueeze(0).unsqueeze(0)
            else:
                res['spherical_coeffs'][i][key] *= 0
                res['radial_width'][i][key] *= 0
                res['radial_scale'][i][key] *= 0

    res = model.property_models['density'](res)

    ml_integral.append(torch.sum(res['density'] * sample['coord_weights']).detach().cpu().numpy())
    ml_loss.append((torch.sum(torch.abs(sample['density'] - res['density']) * sample['coord_weights'])/torch.sum(sample['atom_numbers'])).detach().cpu().numpy())

print('pred density integral', np.mean(ml_integral))
print('density loss', np.mean(ml_loss))
print(L0_coeffs['radial_width'])

cg_matrix shape torch.Size([121, 121, 121])
args energy_unit_in kcal
args energy_unit_out kcal
conversions in <function kcal_to_kcal at 0x7f19934c98c8>
conversions out <function kcal_to_kcal at 0x7f19934c98c8>
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
creating embedding
init_coeffs None
orbital basis {6: [(6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 4), (6, 1, 4), (6, 1, 4), (6, 1, 5), (6, 1, 5)], 8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1

In [67]:
args.restart = None
args.integral_constraint = None 
args.scale_sph_degrees = True
args.expansion_constraint = None 
args.verbose = 0
args.L0_coeffs_file = 'datasets/augccpvqzjkfit_init_L0_free.npy'
L0_coeffs = np.load(args.L0_coeffs_file, allow_pickle=True).item()
model = load_model(args, dataset)
ml_integral = []
ml_loss = []

for i in range(10):
    sample = dataset.get_properties([i])
    res = model(sample)

    for i in range(len(res['spherical_coeffs'])):
        for key in res['spherical_coeffs'][i].keys():
            z = key[0]
            z_s = utils.numbers_to_symbols([z])[0]
            L = key[1]
            if L == 0:
                res['spherical_coeffs'][i][key] = L0_coeffs['spherical_coeffs'][z_s].unsqueeze(0).unsqueeze(0)
                res['radial_width'][i][key] = L0_coeffs['radial_width'][z_s].unsqueeze(0).unsqueeze(0)
                res['radial_scale'][i][key] = L0_coeffs['radial_scale'][z_s].unsqueeze(0).unsqueeze(0)
            else:
                res['spherical_coeffs'][i][key] *= 0
                res['radial_width'][i][key] *= 0
                res['radial_scale'][i][key] *= 0

    res = model.property_models['density'](res)

    ml_integral.append(torch.sum(res['density'] * sample['coord_weights']).detach().cpu().numpy())
    ml_loss.append((torch.sum(torch.abs(sample['density'] - res['density']) * sample['coord_weights'])/torch.sum(sample['atom_numbers'])).detach().cpu().numpy())

print('pred density integral', np.mean(ml_integral))
print('density loss', np.mean(ml_loss))

print(L0_coeffs['radial_width'])

cg_matrix shape torch.Size([121, 121, 121])
args energy_unit_in kcal
args energy_unit_out kcal
conversions in <function kcal_to_kcal at 0x7f19934c98c8>
conversions out <function kcal_to_kcal at 0x7f19934c98c8>
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
creating embedding
init_coeffs None
orbital basis {6: [(6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 4), (6, 1, 4), (6, 1, 4), (6, 1, 5), (6, 1, 5)], 8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1